# TabPFN-Client Versuch: Ordinaler Regressor auf Validierung 2023

Dieses Notebook testet den API-Zugriff über `tabpfn_client.TabPFNRegressor`. Es orientiert sich an `12-02`: Training wird aus vollstaendigen Stages bis einschliesslich 2022 gezogen, validiert wird auf dem kompletten Jahr 2023. Das Ziel ist `target_ordinal`, abgeleitet aus `rank`.

Die API-Konfiguration ist bewusst klein gehalten: `target_rows=5_000`, `seed=42`, `n_estimators=1`. Am Ende wird der gemessene Verbrauch mit einer Hochrechnung fuer den Zielanwendungsfall `<=2023 -> 2024+2025` zusammengefasst.

## Konfiguration

`target_ordinal` ist nur in diesem Notebook definiert:

- `rank` 1-5: 3
- `rank` 6-10: 2
- `rank` 11-20: 1
- Rest: 0

## Notebook-Setup

Diese Zelle sammelt alle Pfade, Features und Laufparameter an einer Stelle. Wichtig fuer diesen Versuch sind `TARGET = "target_ordinal"`, `TRAIN_TARGET_ROWS = 5_000`, `SEED = 42` und `N_ESTIMATORS = 1`. Der API-Lauf ist bewusst von der lokalen Vorbereitung getrennt, damit die Plausibilitaetschecks ohne Credit-Verbrauch ausgefuehrt werden koennen.

In [1]:
# Grundsetup: Imports, Pfade und feste Parameter fuer diesen Versuch.
# Diese Werte steuern nur den Notebook-Lauf; Tokens werden hier nicht gespeichert.
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error

try:
    from IPython.display import display
except ImportError:
    display = print

DATA_DIR = Path("../../data/processed")
RESULT_DIR = DATA_DIR / "tabpfn_3_experiments"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_PATH = DATA_DIR / "26_cleaned_master_data.pkl"

TARGET = "target_ordinal"
TRAIN_TARGET_ROWS = 5_000
SEED = 42
N_ESTIMATORS = 1
MODEL_PATH = "auto"
FORCE_RERUN = False
RUN_API = True

DAILY_PREDICTION_TOKEN_LIMIT = 50_000_000
MONTHLY_PREDICTION_TOKEN_LIMIT = 200_000_000

FEATURE_COLUMNS = [
    "distance", "vertical_meters", "stage_nr", "team_tier", "age_at_race",
    "rider_bmi", "wind_stability_index", "weather_temp_mean", "weather_temp_trend",
    "weather_rain_prob_mean", "weather_precipitation_mean", "weather_humidity_mean",
    "gradient_final_km", "lag_rider_points_season", "lag_rider_rank_season",
    "lag_race_competitiveness_median", "lag_team_power_index",
]

META_COLUMNS = ["meta_year", "meta_name", "meta_current_team", "meta_race", "stage_nr", "stage_id"]
RUN_LABEL = f"tabpfn_client_reg_ordinal_val2023_train_{TRAIN_TARGET_ROWS}_seed_{SEED}"

print(f"Run label: {RUN_LABEL}")
print(f"Target   : {TARGET}")
print(f"Features : {len(FEATURE_COLUMNS)}")
print(f"n_estimators: {N_ESTIMATORS}")

Run label: tabpfn_client_reg_ordinal_val2023_train_5000_seed_42
Target   : target_ordinal
Features : 17
n_estimators: 1


## Authentifizierung und Usage-Abfrage

Der Client nutzt zuerst `TABPFN_TOKEN` aus der Umgebung. Wenn diese Variable nicht gesetzt ist, startet `tabpfn_client.init()` den normalen Login-/Token-Flow und kann einen bereits gecachten Token wiederverwenden. Die Hilfsfunktionen speichern oder drucken keinen Token; sie erfassen nur Usage-Text, strukturierte Usage-Daten und Modelllimits.

In [2]:
# Hilfsfunktionen fuer tabpfn-client Authentifizierung und Verbrauchsabfrage.
# Der Token wird nur aus der Umgebung gelesen oder interaktiv vom Client bezogen.
def clean_tabpfn_token(raw_token):
    token = str(raw_token).strip()
    for prefix in ("export TABPFN_TOKEN=", "TABPFN_TOKEN="):
        if token.startswith(prefix):
            token = token[len(prefix):].strip()
    if (token.startswith('"') and token.endswith('"')) or (token.startswith("'") and token.endswith("'")):
        token = token[1:-1].strip()
    return token


def ensure_tabpfn_client_ready():
    try:
        import tabpfn_client
        from tabpfn_client import TabPFNRegressor, get_api_usage, init
        from tabpfn_client.client import ServiceClient
        from tabpfn_client.config import Config
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "tabpfn-client fehlt. Bitte zuerst ausfuehren: %pip install -U tabpfn-client"
        ) from exc

    token = clean_tabpfn_token(os.environ.get("TABPFN_TOKEN", ""))
    if token:
        # Direct authorization avoids printing or persisting the token in notebook outputs.
        ServiceClient.authorize(token)
        Config.use_server = True
        Config.is_initialized = True
        print("TABPFN_TOKEN ist aus der Umgebung gesetzt.")
    else:
        print("Kein TABPFN_TOKEN in der Umgebung gefunden. Starte interaktiven tabpfn-client Login.")
        init()

    print(f"tabpfn-client Version: {getattr(tabpfn_client, '__version__', 'unbekannt')}")
    return TabPFNRegressor, get_api_usage, ServiceClient


def get_usage_snapshot(get_api_usage_func=None, service_client=None):
    snapshot = {
        "usage_text": None,
        "raw_usage_json": None,
        "current_usage": np.nan,
        "usage_limit": np.nan,
        "reset_time": None,
        "error": None,
    }
    try:
        if get_api_usage_func is not None:
            snapshot["usage_text"] = get_api_usage_func()
    except Exception as exc:
        snapshot["error"] = f"get_api_usage text failed: {type(exc).__name__}: {exc}"

    try:
        if service_client is not None:
            token = service_client.get_access_token()
            if token:
                raw = service_client.get_api_usage(token)
                snapshot["raw_usage_json"] = json.dumps(raw, sort_keys=True)
                snapshot["current_usage"] = float(raw.get("current_usage", np.nan))
                limit = raw.get("usage_limit", np.nan)
                snapshot["usage_limit"] = np.inf if str(limit) == "-1" else float(limit)
                snapshot["reset_time"] = raw.get("reset_time")
    except Exception as exc:
        if snapshot["error"]:
            snapshot["error"] += f" | structured usage failed: {type(exc).__name__}: {exc}"
        else:
            snapshot["error"] = f"structured usage failed: {type(exc).__name__}: {exc}"
    return snapshot


def get_model_limits_snapshot(service_client):
    try:
        limits = service_client.get_model_limits()
        if limits is None:
            return None
        if hasattr(limits, "model_dump"):
            return limits.model_dump()
        return dict(limits)
    except Exception as exc:
        return {"error": f"{type(exc).__name__}: {exc}"}

## Target, Splits und Sampling

Hier wird `target_ordinal` nur fuer dieses Notebook aus `rank` erzeugt. Danach werden dieselben Jahres-Splits wie in den TabPFN-Notebooks `12-02` bis `12-04` gebaut. Das Training nutzt vollstaendige Stages, damit keine Etappe teilweise in den Trainingskontext kommt.

In [3]:
# Datenaufbereitung: ordinales Target erzeugen und dieselben Split-Regeln wie in 12-02 anwenden.
# Die Funktion make_split_bundle liefert y bewusst als float fuer den Regressor.
def make_target_ordinal(rank_series):
    rank = pd.to_numeric(rank_series, errors="coerce")
    return np.select(
        [
            rank.between(1, 5, inclusive="both"),
            rank.between(6, 10, inclusive="both"),
            rank.between(11, 20, inclusive="both"),
        ],
        [3, 2, 1],
        default=0,
    ).astype(int)


def load_master_data():
    df = pd.read_pickle(MASTER_PATH).copy()
    df["stage_id"] = (
        df["meta_race"].astype(str)
        + "_"
        + df["meta_year"].astype(str)
        + "_ST"
        + df["stage_nr"].astype(str)
    )
    df["target_ordinal"] = make_target_ordinal(df["rank"])
    return df


def make_split_bundle(df, mask):
    split = df.loc[mask].copy().reset_index(drop=True)
    return {
        "X": split[FEATURE_COLUMNS].copy(),
        "y": split[TARGET].astype(float).copy(),
        "target_ordinal": split[TARGET].astype(int).copy(),
        "y_rank": split["rank"].astype(float).copy(),
        "meta": split[META_COLUMNS].copy(),
    }


def load_experiment_splits():
    df = load_master_data()
    return df, {
        "train_selection": make_split_bundle(df, df["meta_year"] <= 2022),
        "validation": make_split_bundle(df, df["meta_year"] == 2023),
        "test_original": make_split_bundle(df, df["meta_year"].isin([2024, 2025])),
        "train_final": make_split_bundle(df, df["meta_year"] <= 2023),
    }


def summarize_split(name, bundle):
    meta = bundle["meta"]
    y = bundle["y"]
    stage_counts = meta.groupby("stage_id").size()
    return {
        "split": name,
        "rows": int(len(meta)),
        "features": int(bundle["X"].shape[1]),
        "years": ", ".join(map(str, sorted(meta["meta_year"].unique()))),
        "n_stages": int(meta["stage_id"].nunique()),
        "stage_rows_min": int(stage_counts.min()),
        "stage_rows_median": float(stage_counts.median()),
        "stage_rows_mean": float(stage_counts.mean()),
        "stage_rows_max": int(stage_counts.max()),
        "target_mean": float(y.mean()),
    }


def make_stage_preserving_subset(X, y, meta, target_rows, seed):
    stage_counts = meta.groupby("stage_id", sort=False).size().reset_index(name="rows")
    rng = np.random.default_rng(seed)
    shuffled_stage_ids = stage_counts["stage_id"].to_numpy(copy=True)
    rng.shuffle(shuffled_stage_ids)

    selected_stage_ids = []
    selected_rows = 0
    row_lookup = stage_counts.set_index("stage_id")["rows"].to_dict()

    for stage_id in shuffled_stage_ids:
        selected_stage_ids.append(stage_id)
        selected_rows += int(row_lookup[stage_id])
        if selected_rows >= target_rows:
            break

    selected_stage_ids = set(selected_stage_ids)
    mask = meta["stage_id"].isin(selected_stage_ids)

    X_sub = X.loc[mask].reset_index(drop=True)
    y_sub = y.loc[mask].reset_index(drop=True)
    meta_sub = meta.loc[mask].reset_index(drop=True)

    validate_stage_preserving_sample(meta, meta_sub, target_rows)

    sample_info = {
        "target_rows": int(target_rows),
        "actual_rows": int(len(meta_sub)),
        "row_overshoot": int(len(meta_sub) - target_rows),
        "n_stages": int(meta_sub["stage_id"].nunique()),
        "seed": int(seed),
        "target_mean": float(y_sub.mean()),
    }
    return X_sub, y_sub, meta_sub, sample_info


def validate_stage_preserving_sample(source_meta, sample_meta, target_rows):
    if len(sample_meta) < target_rows:
        raise AssertionError(f"Sample hat nur {len(sample_meta)} Zeilen, Ziel war mindestens {target_rows}.")

    source_counts = source_meta.groupby("stage_id").size()
    sample_counts = sample_meta.groupby("stage_id").size()

    mismatched = []
    for stage_id, sample_count in sample_counts.items():
        source_count = int(source_counts.loc[stage_id])
        if int(sample_count) != source_count:
            mismatched.append((stage_id, int(sample_count), source_count))

    if mismatched:
        raise AssertionError(f"Teilweise Stage-Auswahl gefunden: {mismatched[:5]}")
    return True

## Evaluationslogik

Die Regressor-Ausgabe ist ein kontinuierlicher Score fuer `target_ordinal`. Fuer Ordinalmetriken wird dieser Score direkt genutzt; fuer Accuracy wird er auf den Bereich 0 bis 3 gerundet. Fuer Rankingmetriken wird innerhalb jeder Stage absteigend nach dem Score sortiert.

In [4]:
# Evaluation: globale Ordinalmetriken und stage-basierte Rankingmetriken.
# Hoehere vorhergesagte Ordinalwerte werden als bessere Platzierung interpretiert.
def safe_spearman(a, b):
    a = pd.Series(a, dtype="float64")
    b = pd.Series(b, dtype="float64")
    valid = a.notna() & b.notna()
    if valid.sum() < 2 or a[valid].nunique() < 2 or b[valid].nunique() < 2:
        return np.nan
    return float(a[valid].corr(b[valid], method="spearman"))


def dcg_at_k(relevance, k):
    rel = np.asarray(relevance, dtype=float)[:k]
    if len(rel) == 0:
        return np.nan
    gains = np.power(2.0, rel) - 1.0
    discounts = np.log2(np.arange(len(rel)) + 2.0)
    return float(np.sum(gains / discounts))


def ndcg_at_k_for_stage(stage_pred, k):
    predicted = stage_pred.sort_values(
        ["predicted_target_ordinal_raw", "_row_order"],
        ascending=[False, True],
        kind="mergesort",
    )
    ideal = stage_pred.sort_values(
        ["actual_target_ordinal", "_row_order"],
        ascending=[False, True],
        kind="mergesort",
    )
    idcg = dcg_at_k(ideal["actual_target_ordinal"].to_numpy(), k)
    if not np.isfinite(idcg) or idcg <= 0:
        return np.nan
    return dcg_at_k(predicted["actual_target_ordinal"].to_numpy(), k) / idcg


def build_prediction_frame(meta, y_rank, y_true_ordinal, predicted_ordinal_raw):
    pred = meta.reset_index(drop=True).copy()
    pred["_row_order"] = np.arange(len(pred))
    pred["actual_rank"] = y_rank.reset_index(drop=True).astype(float)
    pred["actual_target_ordinal"] = y_true_ordinal.reset_index(drop=True).astype(int)
    pred["predicted_target_ordinal_raw"] = np.asarray(predicted_ordinal_raw, dtype=float)
    pred["predicted_target_ordinal_rounded"] = (
        np.rint(pred["predicted_target_ordinal_raw"]).clip(0, 3).astype(int)
    )
    pred["absolute_ordinal_error"] = (
        pred["predicted_target_ordinal_raw"] - pred["actual_target_ordinal"]
    ).abs()
    pred["rank_within_stage_by_model"] = (
        pred.groupby("stage_id")["predicted_target_ordinal_raw"]
        .rank(ascending=False, method="first")
        .astype(int)
    )
    return pred


def get_stage_top_n_predictions(pred, n):
    return (
        pred.sort_values(
            ["stage_id", "predicted_target_ordinal_raw", "_row_order"],
            ascending=[True, False, True],
            kind="mergesort",
        )
        .groupby("stage_id", as_index=False)
        .head(n)
        .reset_index(drop=True)
    )


def evaluate_prediction_frame(pred):
    y_true = pred["actual_target_ordinal"].astype(int)
    y_pred_raw = pred["predicted_target_ordinal_raw"].astype(float)
    y_pred_round = pred["predicted_target_ordinal_rounded"].astype(int)

    metrics = {
        "rows": int(len(pred)),
        "n_stages": int(pred["stage_id"].nunique()),
        "ordinal_mae": float(mean_absolute_error(y_true, y_pred_raw)),
        "ordinal_rmse": float(mean_squared_error(y_true, y_pred_raw) ** 0.5),
        "ordinal_accuracy_rounded": float(accuracy_score(y_true, y_pred_round)),
        "spearman_global": safe_spearman(y_true, y_pred_raw),
    }

    stage_rows = []
    for stage_id, stage_pred in pred.groupby("stage_id", sort=False):
        stage_metrics = {"stage_id": stage_id}
        stage_metrics["stage_spearman"] = safe_spearman(
            stage_pred["actual_target_ordinal"],
            stage_pred["predicted_target_ordinal_raw"],
        )
        for k in [5, 10, 20]:
            stage_metrics[f"ndcg_at_{k}"] = ndcg_at_k_for_stage(stage_pred, k)
        for n in [1, 5, 10, 20]:
            top_n = get_stage_top_n_predictions(stage_pred, n)
            stage_metrics[f"top_{n}_accuracy"] = bool((top_n["actual_rank"] == 1).any())

        top_10 = get_stage_top_n_predictions(stage_pred, 10)
        actual_top10_count = int((stage_pred["actual_target_ordinal"] >= 2).sum())
        if actual_top10_count > 0:
            stage_metrics["stage_top10_recall"] = float(
                (top_10["actual_target_ordinal"] >= 2).sum() / actual_top10_count
            )
        else:
            stage_metrics["stage_top10_recall"] = np.nan
        stage_rows.append(stage_metrics)

    stage_metrics_df = pd.DataFrame(stage_rows)
    for k in [5, 10, 20]:
        metrics[f"ndcg_at_{k}"] = float(stage_metrics_df[f"ndcg_at_{k}"].mean())
    for n in [1, 5, 10, 20]:
        metrics[f"top_{n}_accuracy"] = float(stage_metrics_df[f"top_{n}_accuracy"].mean())
    metrics["mean_stage_spearman"] = float(stage_metrics_df["stage_spearman"].mean())
    metrics["mean_stage_top10_recall"] = float(stage_metrics_df["stage_top10_recall"].mean())
    return metrics, stage_metrics_df

## Verbrauchsschaetzung

Die API-Usage wird vor und nach dem Lauf gelesen. Daraus entsteht eine gemessene Delta fuer diesen Versuch. Zusaetzlich berechnet das Notebook eine einfache lineare Hochrechnung und die dokumentierte v2.x-Vergleichsformel; fuer v3 ist die gemessene Delta aussagekraeftiger, weil die Kosten sublinear sein koennen.

In [5]:
# Verbrauchsschaetzung: gemessene API-Nutzung plus lineare und v2.x-basierte Vergleichswerte.
# Fuer TabPFN v3 bleibt die gemessene Delta der wichtigste Wert.
def v2_formula_tokens(train_rows, test_rows, n_columns, n_estimators):
    return int(max(5_000, (int(train_rows) + int(test_rows)) * int(n_columns) * int(n_estimators)))


def finite_or_nan(value):
    try:
        value = float(value)
    except Exception:
        return np.nan
    return value if np.isfinite(value) else np.nan


def pool_percent(tokens, pool_size):
    tokens = finite_or_nan(tokens)
    if not np.isfinite(tokens) or tokens <= 0:
        return np.nan
    return float(tokens / pool_size)


def runs_until_pool_exhausted(tokens, pool_size):
    tokens = finite_or_nan(tokens)
    if not np.isfinite(tokens) or tokens <= 0:
        return np.nan
    return float(pool_size / tokens)


def build_usage_estimate_table(
    before_usage,
    after_usage,
    actual_train_rows,
    validation_rows,
    final_train_rows,
    future_test_rows,
    n_columns,
    n_estimators,
):
    before_current = finite_or_nan(before_usage.get("current_usage"))
    after_current = finite_or_nan(after_usage.get("current_usage"))
    measured_delta = after_current - before_current if np.isfinite(before_current) and np.isfinite(after_current) else np.nan
    if np.isfinite(measured_delta) and measured_delta < 0:
        measured_delta = np.nan

    current_v2 = v2_formula_tokens(actual_train_rows, validation_rows, n_columns, n_estimators)
    intended_v2 = v2_formula_tokens(final_train_rows, future_test_rows, n_columns, n_estimators)

    current_row_sum = actual_train_rows + validation_rows
    intended_row_sum = final_train_rows + future_test_rows
    linear_extrapolated = (
        measured_delta * intended_row_sum / current_row_sum
        if np.isfinite(measured_delta) and current_row_sum > 0
        else np.nan
    )

    rows = [
        {
            "scenario": "measured_current_run_delta",
            "train_rows": int(actual_train_rows),
            "test_rows": int(validation_rows),
            "n_columns": int(n_columns),
            "n_estimators": int(n_estimators),
            "estimated_tokens": measured_delta,
            "basis": "API usage after - before",
        },
        {
            "scenario": "current_run_v2_formula",
            "train_rows": int(actual_train_rows),
            "test_rows": int(validation_rows),
            "n_columns": int(n_columns),
            "n_estimators": int(n_estimators),
            "estimated_tokens": current_v2,
            "basis": "documented v2.x formula",
        },
        {
            "scenario": "intended_2024_2025_linear_from_measured",
            "train_rows": int(final_train_rows),
            "test_rows": int(future_test_rows),
            "n_columns": int(n_columns),
            "n_estimators": int(n_estimators),
            "estimated_tokens": linear_extrapolated,
            "basis": "linear row extrapolation from measured API delta",
        },
        {
            "scenario": "intended_2024_2025_v2_formula",
            "train_rows": int(final_train_rows),
            "test_rows": int(future_test_rows),
            "n_columns": int(n_columns),
            "n_estimators": int(n_estimators),
            "estimated_tokens": intended_v2,
            "basis": "documented v2.x formula",
        },
    ]
    summary = pd.DataFrame(rows)
    summary["daily_limit_tokens"] = DAILY_PREDICTION_TOKEN_LIMIT
    summary["monthly_limit_tokens"] = MONTHLY_PREDICTION_TOKEN_LIMIT
    summary["share_of_daily_limit"] = summary["estimated_tokens"].apply(
        lambda x: pool_percent(x, DAILY_PREDICTION_TOKEN_LIMIT)
    )
    summary["share_of_monthly_limit"] = summary["estimated_tokens"].apply(
        lambda x: pool_percent(x, MONTHLY_PREDICTION_TOKEN_LIMIT)
    )
    summary["equivalent_runs_until_daily_limit"] = summary["estimated_tokens"].apply(
        lambda x: runs_until_pool_exhausted(x, DAILY_PREDICTION_TOKEN_LIMIT)
    )
    summary["equivalent_runs_until_monthly_limit"] = summary["estimated_tokens"].apply(
        lambda x: runs_until_pool_exhausted(x, MONTHLY_PREDICTION_TOKEN_LIMIT)
    )
    summary["api_usage_before_text"] = before_usage.get("usage_text")
    summary["api_usage_after_text"] = after_usage.get("usage_text")
    summary["api_usage_before_raw"] = before_usage.get("raw_usage_json")
    summary["api_usage_after_raw"] = after_usage.get("raw_usage_json")
    summary["usage_error_before"] = before_usage.get("error")
    summary["usage_error_after"] = after_usage.get("error")
    summary["note"] = (
        "TabPFN v3 nutzt laut Doku sublineare Metering-Kosten; die gemessene API-Delta "
        "ist fuer v3 verlaesslicher als lineare Hochrechnungen."
    )
    return summary

## Lokale Pruefung vor dem API-Lauf

Diese Zelle ist der Sicherheitsgurt vor dem Credit-Verbrauch: Sie prueft Target-Grenzen, Split-Groessen und das stage-preserving Sample. Wenn hier ein Assert fehlschlaegt, sollte die API-Zelle nicht ausgefuehrt werden.

In [6]:
# Lokaler Check vor dem API-Lauf: Daten laden, Target pruefen und stage-preserving Sample ziehen.
# Diese Zelle verbraucht keine TabPFN-Credits.
df, splits = load_experiment_splits()
train_bundle = splits["train_selection"]
val_bundle = splits["validation"]
train_final_bundle = splits["train_final"]
test_original_bundle = splits["test_original"]

print("Target distribution:")
display(df[TARGET].value_counts().sort_index().rename("count").to_frame())

boundary_check = pd.DataFrame({"rank": [5, 6, 10, 11, 20, 21]})
boundary_check["target_ordinal"] = make_target_ordinal(boundary_check["rank"])
print("Boundary check:")
display(boundary_check)

split_summary = pd.DataFrame([
    summarize_split("train_selection <=2022", train_bundle),
    summarize_split("validation 2023", val_bundle),
    summarize_split("train_final <=2023", train_final_bundle),
    summarize_split("test_original 2024+2025", test_original_bundle),
])
display(split_summary)

X_fit, y_fit, meta_fit, sample_info = make_stage_preserving_subset(
    train_bundle["X"],
    train_bundle["y"],
    train_bundle["meta"],
    target_rows=TRAIN_TARGET_ROWS,
    seed=SEED,
)

print(
    f"Fit-Set: {sample_info['actual_rows']:,} Zeilen aus {sample_info['n_stages']} Stages; "
    f"Overshoot: {sample_info['row_overshoot']:,}; Target-Mittel: {sample_info['target_mean']:.4f}"
)
print(f"Validation 2023: {val_bundle['X'].shape}")

assert TARGET == "target_ordinal"
assert set(df[TARGET].unique()).issubset({0, 1, 2, 3})
expected_boundary = [3, 2, 2, 1, 1, 0]
assert boundary_check["target_ordinal"].tolist() == expected_boundary
assert train_bundle["X"].shape == (169_349, 17)
assert val_bundle["X"].shape == (8_897, 17)
assert sample_info["actual_rows"] == 5_144
assert sample_info["n_stages"] == 30
selected_stage_ids = set(meta_fit["stage_id"])
selected_mask = train_bundle["meta"]["stage_id"].isin(selected_stage_ids)
expected_y_fit = train_bundle["y"].loc[selected_mask].reset_index(drop=True)
assert y_fit.reset_index(drop=True).equals(expected_y_fit)

print("Lokale Checks bestanden.")

Target distribution:


,count
target_ordinal,
0,172740
1,11628
2,5829
3,5851


Boundary check:


,rank,target_ordinal
0,5,3
1,6,2
2,10,2
3,11,1
4,20,1
5,21,0


,split,rows,features,years,n_stages,stage_rows_min,stage_rows_median,stage_rows_mean,stage_rows_max,target_mean
0,train_selection <=2022,169349,17,"2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012...",997,117,171.0,169.858576,206,0.206568
1,validation 2023,8897,17,2023,57,122,158.0,156.087719,175,0.223896
2,train_final <=2023,178246,17,"2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012...",1054,117,170.0,169.113852,206,0.207432
3,test_original 2024+2025,17802,17,"2024, 2025",112,131,161.5,158.946429,179,0.217110


Fit-Set: 5,144 Zeilen aus 30 Stages; Overshoot: 144; Target-Mittel: 0.2057
Validation 2023: (8897, 17)
Lokale Checks bestanden.


## API-Lauf

Diese Zelle fuehrt den eigentlichen TabPFN-Client-Lauf aus. Sie nutzt `target_ordinal` aus `train_bundle["y"]` als Regressionsziel. Ein Token wird nur aus der Umgebung gelesen oder ueber den interaktiven Login bezogen; es wird nicht im Notebook gespeichert oder ausgegeben.

In [7]:
# API-Lauf: nur diese Zelle ruft tabpfn-client fit/predict auf und kann Credits verbrauchen.
# Bei vorhandenen Prediction-Dateien wird ohne FORCE_RERUN aus dem Cache gelesen.
pred_path = RESULT_DIR / f"{RUN_LABEL}_predictions.csv"
stage_metrics_path = RESULT_DIR / f"{RUN_LABEL}_stage_metrics.csv"
metrics_path = RESULT_DIR / f"{RUN_LABEL}_metrics.csv"
usage_path = RESULT_DIR / f"{RUN_LABEL}_usage.csv"
model_limits_path = RESULT_DIR / f"{RUN_LABEL}_model_limits.json"

if pred_path.exists() and not FORCE_RERUN:
    print(f"Lade vorhandene Predictions: {pred_path}")
    pred = pd.read_csv(pred_path)
    fit_seconds = np.nan
    predict_seconds = np.nan
    source = "cached"
    before_usage = get_usage_snapshot()
    after_usage = get_usage_snapshot()
else:
    if not RUN_API:
        raise RuntimeError("RUN_API=False. Setze RUN_API=True, um den tabpfn-client API-Lauf zu starten.")

    TabPFNRegressor, get_api_usage_func, ServiceClient = ensure_tabpfn_client_ready()
    model_limits = get_model_limits_snapshot(ServiceClient)
    model_limits_path.write_text(json.dumps(model_limits, indent=2, sort_keys=True), encoding="utf-8")

    before_usage = get_usage_snapshot(get_api_usage_func, ServiceClient)

    reg = TabPFNRegressor(
        model_path=MODEL_PATH,
        n_estimators=N_ESTIMATORS,
        random_state=SEED,
    )

    start_fit = time.time()
    reg.fit(
        X_fit,
        y_fit,
        description=(
            "GADA cycling stage prediction ordinal regression; "
            "train <=2022 stage-preserving min 5000 rows seed 42; validation 2023"
        ),
    )
    fit_seconds = time.time() - start_fit

    start_pred = time.time()
    predicted_ordinal_raw = reg.predict(val_bundle["X"], output_type="mean")
    predict_seconds = time.time() - start_pred

    after_usage = get_usage_snapshot(get_api_usage_func, ServiceClient)

    pred = build_prediction_frame(
        val_bundle["meta"],
        val_bundle["y_rank"],
        val_bundle["target_ordinal"],
        predicted_ordinal_raw,
    )
    pred.drop(columns=["_row_order"]).to_csv(pred_path, index=False)
    source = "computed"

if "_row_order" not in pred.columns:
    pred["_row_order"] = np.arange(len(pred))

metrics, stage_metrics = evaluate_prediction_frame(pred)
metrics.update({
    "run_label": RUN_LABEL,
    "phase": "validation_2023_tabpfn_client_ordinal_regressor",
    "target": TARGET,
    "model_path": MODEL_PATH,
    "n_estimators": N_ESTIMATORS,
    "target_rows": TRAIN_TARGET_ROWS,
    "actual_train_rows": sample_info["actual_rows"],
    "row_overshoot": sample_info["row_overshoot"],
    "train_stages": sample_info["n_stages"],
    "seed": SEED,
    "fit_seconds": fit_seconds,
    "predict_seconds": predict_seconds,
    "source": source,
})

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(metrics_path, index=False)
stage_metrics.to_csv(stage_metrics_path, index=False)

usage_summary = build_usage_estimate_table(
    before_usage=before_usage,
    after_usage=after_usage,
    actual_train_rows=sample_info["actual_rows"],
    validation_rows=len(val_bundle["X"]),
    final_train_rows=len(train_final_bundle["X"]),
    future_test_rows=len(test_original_bundle["X"]),
    n_columns=len(FEATURE_COLUMNS),
    n_estimators=N_ESTIMATORS,
)
usage_summary.to_csv(usage_path, index=False)

display(metrics_df.T.rename(columns={0: "value"}))
display(usage_summary)
print(f"Predictions: {pred_path}")
print(f"Metrics    : {metrics_path}")
print(f"Usage      : {usage_path}")

Kein TABPFN_TOKEN in der Umgebung gefunden. Starte interaktiven tabpfn-client Login.


Found existing access token, reusing it for authentication.

tabpfn-client Version: unbekannt
00:03 Fitting... Done!
00:22 Predicting... Done!


,value
rows,8897
n_stages,57
ordinal_mae,0.309594
ordinal_rmse,0.615225
ordinal_accuracy_rounded,0.8495
spearman_global,0.285543
ndcg_at_5,0.372195
ndcg_at_10,0.387078
ndcg_at_20,0.418057
top_1_accuracy,0.175439


,scenario,train_rows,test_rows,n_columns,n_estimators,estimated_tokens,basis,daily_limit_tokens,monthly_limit_tokens,share_of_daily_limit,share_of_monthly_limit,equivalent_runs_until_daily_limit,equivalent_runs_until_monthly_limit,api_usage_before_text,api_usage_after_text,api_usage_before_raw,api_usage_after_raw,usage_error_before,usage_error_after,note
0,measured_current_run_delta,5144,8897,17,1,1.000400e+04,API usage after - before,50000000,200000000,0.000200,0.000050,4998.000800,19992.003199,"Currently, you have used 0 of the allowed limi...","Currently, you have used 10004 of the allowed ...","{""current_usage"": 0, ""daily_limit"": 100000000,...","{""current_usage"": 10004, ""daily_limit"": 100000...",None,None,TabPFN v3 nutzt laut Doku sublineare Metering-...
1,current_run_v2_formula,5144,8897,17,1,2.386970e+05,documented v2.x formula,50000000,200000000,0.004774,0.001193,209.470584,837.882336,"Currently, you have used 0 of the allowed limi...","Currently, you have used 10004 of the allowed ...","{""current_usage"": 0, ""daily_limit"": 100000000,...","{""current_usage"": 10004, ""daily_limit"": 100000...",None,None,TabPFN v3 nutzt laut Doku sublineare Metering-...
2,intended_2024_2025_linear_from_measured,178246,17802,17,1,1.396812e+05,linear row extrapolation from measured API delta,50000000,200000000,0.002794,0.000698,357.957894,1431.831577,"Currently, you have used 0 of the allowed limi...","Currently, you have used 10004 of the allowed ...","{""current_usage"": 0, ""daily_limit"": 100000000,...","{""current_usage"": 10004, ""daily_limit"": 100000...",None,None,TabPFN v3 nutzt laut Doku sublineare Metering-...
3,intended_2024_2025_v2_formula,178246,17802,17,1,3.332816e+06,documented v2.x formula,50000000,200000000,0.066656,0.016664,15.002328,60.009313,"Currently, you have used 0 of the allowed limi...","Currently, you have used 10004 of the allowed ...","{""current_usage"": 0, ""daily_limit"": 100000000,...","{""current_usage"": 10004, ""daily_limit"": 100000...",None,None,TabPFN v3 nutzt laut Doku sublineare Metering-...


Predictions: ../../data/processed/tabpfn_3_experiments/tabpfn_client_reg_ordinal_val2023_train_5000_seed_42_predictions.csv
Metrics    : ../../data/processed/tabpfn_3_experiments/tabpfn_client_reg_ordinal_val2023_train_5000_seed_42_metrics.csv
Usage      : ../../data/processed/tabpfn_3_experiments/tabpfn_client_reg_ordinal_val2023_train_5000_seed_42_usage.csv


## Hinweise zur Verbrauchsschaetzung

Die v2.x-Formel `max(5_000, (train_rows + test_rows) * columns * n_estimators)` ist nur eine Vergleichsbasis. TabPFN v3 nutzt laut Prior-Labs-Dokumentation eine sublineare Kostenfunktion, deshalb ist die gemessene API-Delta aus `get_api_usage()` fuer diesen Lauf die wichtigste Zahl.